# POO — Démo comparative

Ce notebook illustre le passage de `extract_api.py` (fonctionnel) à `extract_api_oop.py` (orienté objet), sur le même besoin : extraire des films populaires depuis TMDB.

Pré-requis : un fichier `.env` à la racine du projet contenant `TMDB_API_KEY=votre_cle`.

In [1]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("TMDB_API_KEY")
BASE_URL = "https://api.themoviedb.org/3"

assert API_KEY, "Clé API manquante : vérifiez votre fichier .env"

## 1. Rappel : la version fonctionnelle

Dans `extract_api.py`, la correspondance des genres est une fonction indépendante. Chaque fonction qui en a besoin (ex. `extraire_champs_utiles`) doit se la faire **passer en paramètre** : rien ne la « retient » entre deux appels.

In [2]:
def get_genre_mapping_fonction():
    url = f"{BASE_URL}/genre/movie/list"
    params = {"api_key": API_KEY, "language": "fr-FR"}
    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()
    genres = response.json().get("genres", [])
    print("Appel réseau effectué")  # pour visualiser QUAND l'appel a vraiment lieu
    return {g["id"]: g["name"] for g in genres}

# Chaque appel refait la requête réseau, même si rien n'a changé entre les deux
mapping_1 = get_genre_mapping_fonction()
mapping_2 = get_genre_mapping_fonction()

Appel réseau effectué
Appel réseau effectué


Deux `"Appel réseau effectué"` s'affichent : la fonction ne sait pas qu'elle a déjà fait ce travail. C'est exactement le genre de situation où la POO apporte quelque chose de concret : un objet peut **retenir un état** (ici, le résultat déjà calculé).

## 2. Une première classe minimale

Rappel de vocabulaire :
- `class` : le plan de construction
- `__init__` : ce qui se passe à la création d'un objet (le constructeur)
- `self` : l'objet lui-même, permet d'accéder à ses propres attributs/méthodes
- attribut : une donnée stockée dans l'objet (`self.api_key`)
- méthode : une fonction qui appartient à la classe

In [2]:
class TMDBExtractorDemo:
    BASE_URL = "https://api.themoviedb.org/3"

    def __init__(self, api_key, langue="fr-FR"):
        self.api_key = api_key
        self.langue = langue
        self._genres_mapping = None  # rien n'est encore chargé à la création

    def get_genre_mapping(self):
        if self._genres_mapping is not None:
            print("Résultat déjà en cache, pas d'appel réseau")
            return self._genres_mapping

        url = f"{self.BASE_URL}/genre/movie/list"
        params = {"api_key": self.api_key, "language": self.langue}
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        genres = response.json().get("genres", [])
        print("Appel réseau effectué")
        self._genres_mapping = {g["id"]: g["name"] for g in genres}
        return self._genres_mapping

## 3. Instancier et observer le cache en action

In [4]:
extracteur = TMDBExtractorDemo(API_KEY)

In [5]:
extracteur.api_key

'76c9319204fae873b7b073c60eaf35b1'

In [6]:
extracteur.langue

'fr-FR'

In [ ]:
extracteur = TMDBExtractorDemo(API_KEY)

print("Premier appel :")
mapping = extracteur.get_genre_mapping()


Premier appel :
Appel réseau effectué


In [8]:
print(f"Mapping des genres : {mapping}")

Mapping des genres : {28: 'Action', 12: 'Aventure', 16: 'Animation', 35: 'Comédie', 80: 'Crime', 99: 'Documentaire', 18: 'Drame', 10751: 'Familial', 14: 'Fantastique', 36: 'Histoire', 27: 'Horreur', 10402: 'Musique', 9648: 'Mystère', 10749: 'Romance', 878: 'Science-Fiction', 10770: 'Téléfilm', 53: 'Thriller', 10752: 'Guerre', 37: 'Western'}


In [9]:
print("\nDeuxième appel :")
mapping = extracteur.get_genre_mapping()  # cette fois, pas d'appel réseau


Deuxième appel :
Résultat déjà en cache, pas d'appel réseau


## 4. Deux instances = deux configurations indépendantes

Autre bénéfice concret : on peut créer plusieurs objets avec des configurations différentes, sans qu'ils interfèrent entre eux — impossible aussi simplement avec des fonctions et des variables globales.

In [10]:
extracteur_fr = TMDBExtractorDemo(API_KEY, langue="fr-FR")
extracteur_en = TMDBExtractorDemo(API_KEY, langue="en-US")

genres_fr = extracteur_fr.get_genre_mapping()
genres_en = extracteur_en.get_genre_mapping()

print(list(genres_fr.values())[:3])
print(list(genres_en.values())[:3])

# Chaque objet garde sa propre langue et son propre cache, indépendamment de l'autre

Appel réseau effectué
Appel réseau effectué
['Action', 'Aventure', 'Animation']
['Action', 'Adventure', 'Animation']


## 5. Vers la classe complète

La classe `TMDBExtractor` dans `src/extract_api_oop.py` reprend ce principe et y ajoute toutes les méthodes déjà connues (`get_popular_movies`, `get_movie_details`, `get_movie_credits`, `extraire_champs_utiles`...). Comparez les deux fichiers `extract_api.py` et `extract_api_oop.py` côte à côte : même logique, organisation différente.

In [6]:
import sys
sys.path.append("../src")
from extract_api_oop import TMDBExtractor

extractor = TMDBExtractor(API_KEY)
films = extractor.extraire_et_simplifier_populaires(page=1)

for film in films[:5]:
    print(f"- {film['titre']} ({film['date_sortie']}) — {film['genres']}")

- Spider-Man : Brand New Day (2026-07-29) — ['Science-Fiction', 'Action', 'Aventure']
- Resident Evil (2026-09-16) — ['Horreur', 'Science-Fiction', 'Aventure']
- La Fin d'Oak Street (2026-08-12) — ['Science-Fiction', 'Mystère', 'Thriller']
- Coyote vs. Acme (2026-08-20) — ['Comédie', 'Aventure', 'Familial']
- L'Odyssée (2026-07-15) — ['Aventure', 'Action', 'Fantastique']


---
**À retenir :** la version fonctionnelle (`extract_api.py`) reste tout à fait valable pour ce projet — elle n'est pas "moins bonne". La version objet devient intéressante quand on a besoin de **garder un état** (cache, configuration multiple) ou quand le projet grossit. C'est une porte ouverte vers la Phase 5, pas une obligation cette semaine.